# Deep Learning 基礎講座 最終課題: VQA

画像と質問から回答を予測するタスク。
本 Notebook は学習〜推論〜提出ファイル作成までを 1 ファイルにまとめた提出用ファイルです。

実行すると以下を生成します。
- `submission.npy` : テストデータ(valid.json)に対する予測
- `model.pt` : 予測に使用したモデルの重み
- `submission.zip` : 上記 + 本 Notebook をまとめた提出用 zip


## 1. import

In [ ]:
import os
import re
import time
import random
from statistics import mode
from zipfile import ZipFile

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms


## 2. 設定 (config)

実験設定はここを変更する。

In [ ]:
SEED = 42

BATCH_SIZE = 16

NUM_EPOCHS = 1

LR = 1e-3

WEIGHT_DECAY = 1e-5

OPTIMIZER = "adam"

LOSS_TYPE = "hard"

IMAGE_SIZE = 224

MODEL = "baseline"
EXP_NAME = "baseline"

MODEL_PATH = "./outputs/checkpoints/best_model.pt"
NOTEBOOK_PATH = "./DL_Basic_2026_Spring_competition_VQA.ipynb"

## 3. utils

In [ ]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

## 4. データセット

In [ ]:
def process_text(text):
    """
    入力文と回答のフォーマットを統一するための関数．

    Parameters
    ----------
    text : str
        入力文，もしくは回答．
    """
    # lowercase
    text = text.lower()

    # 数詞を数字に変換
    num_word_to_digit = {
        'zero': '0', 'one': '1', 'two': '2', 'three': '3', 'four': '4',
        'five': '5', 'six': '6', 'seven': '7', 'eight': '8', 'nine': '9',
        'ten': '10'
    }
    for word, digit in num_word_to_digit.items():
        text = text.replace(word, digit)

    # 小数点のピリオドを削除
    text = re.sub(r'(?<!\d)\.(?!\d)', '', text)

    # 冠詞の削除
    text = re.sub(r'\b(a|an|the)\b', '', text)

    # 短縮形のカンマの追加
    contractions = {
        "dont": "don't", "isnt": "isn't", "arent": "aren't", "wont": "won't",
        "cant": "can't", "wouldnt": "wouldn't", "couldnt": "couldn't"
    }
    for contraction, correct in contractions.items():
        text = text.replace(contraction, correct)

    # 句読点をスペースに変換
    text = re.sub(r"[^\w\s':]", ' ', text)

    # 句読点をスペースに変換
    text = re.sub(r'\s+,', ',', text)

    # 連続するスペースを1つに変換
    text = re.sub(r'\s+', ' ', text).strip()

    return text

class VQADataset(torch.utils.data.Dataset):
    """
    VQA データセットを扱うためのクラス．
    """
    def __init__(self, df_path, image_dir, transform=None, answer=True):
        self.transform = transform  # 画像の前処理
        self.image_dir = image_dir  # 画像ファイルのディレクトリ
        self.df = pd.read_json(df_path)  # 画像ファイルのパス，question, answerを持つDataFrame
        self.answer = answer

        # question / answerの辞書を作成
        self.question2idx = {}
        self.answer2idx = {}
        self.idx2question = {}
        self.idx2answer = {}

        # 質問文に含まれる単語を辞書に追加
        for question in self.df["question"]:
            question = process_text(question)
            words = question.split(" ")
            for word in words:
                if word not in self.question2idx:
                    self.question2idx[word] = len(self.question2idx)
        self.idx2question = {v: k for k, v in self.question2idx.items()}  # 逆変換用の辞書(question)

        if self.answer:
            # 回答に含まれる文章を辞書に追加
            for answers in self.df["answers"]:
                for answer in answers:
                    word = answer["answer"]
                    word = process_text(word)
                    if word not in self.answer2idx:
                        self.answer2idx[word] = len(self.answer2idx)
            self.idx2answer = {v: k for k, v in self.answer2idx.items()}  # 逆変換用の辞書(answer)

    def update_dict(self, dataset):
        """
        検証用データ，テストデータの辞書を訓練データの辞書に更新する．

        Parameters
        ----------
        dataset : Dataset
            訓練データのDataset
        """
        self.question2idx = dataset.question2idx
        self.answer2idx = dataset.answer2idx
        self.idx2question = dataset.idx2question
        self.idx2answer = dataset.idx2answer

    def __getitem__(self, idx):
        """
        対応するidxのデータ（画像，質問，回答）を取得．

        Parameters
        ----------
        idx : int
            取得するデータのインデックス

        Returns
        -------
        image : torch.Tensor  (C, H, W)
            画像データ
        question : torch.Tensor  (vocab_size)
            質問文をone-hot表現に変換したもの
        answers : torch.Tensor  (n_answer)
            10人の回答者の回答のid
        mode_answer_idx : torch.Tensor  (1)
            10人の回答者の回答の中で最頻値の回答のid
        """
        image = Image.open(f"{self.image_dir}/{self.df['image'][idx]}").convert("RGB")
        image = self.transform(image)
        question = np.zeros(len(self.idx2question) + 1)  # 未知語用の要素を追加
        question_words = process_text(self.df["question"][idx]).split(" ")
        for word in question_words:
            try:
                question[self.question2idx[word]] = 1  # one-hot表現に変換
            except KeyError:
                question[-1] = 1  # 未知語

        if self.answer:
            answers = [self.answer2idx[process_text(answer["answer"])] for answer in self.df["answers"][idx]]
            mode_answer_idx = mode(answers)  # 最頻値を取得（正解ラベル）

            return {"image": image, "question": torch.Tensor(question),"answers": torch.Tensor(answers), "mode_answer": int(mode_answer_idx),}
        else:
            return {"image": image, "question": torch.Tensor(question)}

    def __len__(self):
        return len(self.df)

## 5. 評価指標

In [ ]:
def VQA_criterion(batch_pred, batch_answers):
    """
    VQA タスクに用いられる評価関数．
    """
    total_acc = 0.

    for pred, answers in zip(batch_pred, batch_answers):
        acc = 0.
        for i in range(len(answers)):
            num_match = 0
            for j in range(len(answers)):
                if i == j:
                    continue
                if pred == answers[j]:
                    num_match += 1
            acc += min(num_match / 3, 1)
        total_acc += acc / 10

    return total_acc / len(batch_pred)

## 6. モデル (ResNet)

In [ ]:
class BasicBlock(nn.Module):
    """
    ResNet の basic block
    """
    expansion = 1

    def __init__(self, in_channels, out_channels, stride=1):
        """
        コンストラクタ．

        Parameters
        ----------
        in_channles: int
            入力のチャネル数
        out_channels:
            出力のチャネル数
        stride: int
            ストライド
        """
        super().__init__()

        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)

        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride),
                nn.BatchNorm2d(out_channels)
            )

    def forward(self, x):
        """
        順伝播処理

        Parameters
        ----------
        x: torch.Tensor
            ブロックへの入力

        Returns
        -------
        out: torch.Tensor
            ブロックへの出力
        """
        residual = x
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))

        out += self.shortcut(residual)
        out = self.relu(out)

        return out
    

class BottleneckBlock(nn.Module):
    """
    ResNet の bottleneck block
    """
    expansion = 4

    def __init__(self, in_channels, out_channels, stride=1):
        """
        コンストラクタ．

        Parameters
        ----------
        in_channles: int
            入力のチャネル数
        out_channels:
            出力のチャネル数
        stride: int
            ストライド
        """
        super().__init__()

        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=1)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=stride, padding=1)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.conv3 = nn.Conv2d(out_channels, out_channels * self.expansion, kernel_size=1, stride=1)
        self.bn3 = nn.BatchNorm2d(out_channels * self.expansion)
        self.relu = nn.ReLU(inplace=True)

        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels * self.expansion:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels * self.expansion, kernel_size=1, stride=stride),
                nn.BatchNorm2d(out_channels * self.expansion)
            )

    def forward(self, x):
        """
        順伝播処理

        Parameters
        ----------
        x: torch.Tensor
            ブロックへの入力

        Returns
        -------
        out: torch.Tensor
            ブロックへの出力
        """
        residual = x
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.relu(self.bn2(self.conv2(out)))
        out = self.bn3(self.conv3(out))

        out += self.shortcut(residual)
        out = self.relu(out)

        return out

class ResNet(nn.Module):
    """
    ResNet の実装
    """
    def __init__(self, block, layers):
        """
        コンストラクタ．

        Parameters
        ----------
        block: torch.nn.Module
            利用するブロックのクラス (BasicBlock / BottleneckBlock)
        layers: list
            各ブロックの層数
        """
        super().__init__()
        self.in_channels = 64

        self.conv1 = nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3)
        self.bn1 = nn.BatchNorm2d(64)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)

        self.layer1 = self._make_layer(block, layers[0], 64)
        self.layer2 = self._make_layer(block, layers[1], 128, stride=2)
        self.layer3 = self._make_layer(block, layers[2], 256, stride=2)
        self.layer4 = self._make_layer(block, layers[3], 512, stride=2)

        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(512 * block.expansion, 512)

    def _make_layer(self, block, blocks, out_channels, stride=1):
        """
        同じ構成を繰り返す部分を生成する．

        Parameters
        ----------
        block: torch.nn.Module
            利用するブロックのクラス (BasicBlock / BottleneckBlock)
        blocks: int
            層数
        out_channels: int
            出力のチャネル数
        stride: int
            ストライド

        Returns
        -------
        layers: torch.nn.ModuleList
            生成した層
        """
        layers = []
        layers.append(block(self.in_channels, out_channels, stride))
        self.in_channels = out_channels * block.expansion
        for _ in range(1, blocks):
            layers.append(block(self.in_channels, out_channels))

        return nn.Sequential(*layers)

    def forward(self, x):
        """
        順伝播処理

        Parameters
        ----------
        x: torch.Tensor
            入力データ

        Returns
        -------
        x: torch.Tensor
            ResNet によって生成される特徴量
        """
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.maxpool(x)

        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)

        x = self.avgpool(x)
        x = x.view(x.size(0), -1)
        x = self.fc(x)

        return x
    

def ResNet18():
    """
    ResNet18 を生成する関数．
    """
    return ResNet(BasicBlock, [2, 2, 2, 2])

def ResNet50():
    """
    ResNet50 を生成する関数．
    """
    return ResNet(BottleneckBlock, [3, 4, 6, 3])

## 7. モデル (VQAModel)

In [ ]:
class VQAModel(nn.Module):
    """
    VQA タスクを解くためのモデル例．
    """
    def __init__(self, vocab_size: int, n_answer: int):
        """
        コンストラクタ．

        Parameters
        ----------
        vocab_size: int
            入力文の語彙数
        n_answer: int
            出力のクラス数
        """
        super().__init__()
        self.resnet = ResNet18()
        self.text_encoder = nn.Linear(vocab_size, 512)

        self.fc = nn.Sequential(
            nn.Linear(1024, 512),
            nn.ReLU(inplace=True),
            nn.Linear(512, n_answer)
        )

    def forward(self, image, question):

        image_feature = self.resnet(image)  # 画像の特徴量
        question_feature = self.text_encoder(question)  # テキストの特徴量

        x = torch.cat([image_feature, question_feature], dim=1)
        x = self.fc(x)

        return x

## 8. 損失関数

In [ ]:
class SoftCrossEntropyLoss(nn.Module):

    def forward(self, logits, soft_targets):

        log_probs = F.log_softmax(logits, dim=1)

        loss = -(soft_targets * log_probs).sum(dim=1)

        return loss.mean()
    
def build_soft_target(answers, num_classes):
    """
    answers:
        (batch_size, 10)

    return:
        (batch_size, num_classes)
    """

    batch_size = answers.shape[0]

    target = torch.zeros(
        batch_size,
        num_classes,
        device=answers.device,
    )

    for i in range(batch_size):
        for ans in answers[i]:
            target[i, int(ans)] += 1

    target /= answers.shape[1]

    return target

## 9. 学習・検証ループ

In [ ]:
def train_one_epoch(model, dataloader, optimizer, criterion, device):
    model.train()

    total_loss = 0
    total_acc = 0
    total_simple_acc = 0

    start = time.time()

    for batch in dataloader:
        image = batch["image"].to(device)
        question = batch["question"].to(device)
        answers = batch["answers"].to(device)
        mode_answer = batch["mode_answer"].to(device)

        pred = model(image, question)

        if LOSS_TYPE == "hard":
            loss = criterion(pred, mode_answer)
        elif LOSS_TYPE == "soft":
            soft_target = build_soft_target(answers, pred.shape[1])
            loss = criterion(pred, soft_target)
        else:
            raise ValueError(f"Unknown LOSS_TYPE: {LOSS_TYPE}")

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        total_acc += VQA_criterion(pred.argmax(1), answers)
        total_simple_acc += (pred.argmax(1) == mode_answer).float().mean().item()

    n = len(dataloader)
    return total_loss / n, total_acc / n, total_simple_acc / n, time.time() - start


@torch.no_grad()
def validate(model, dataloader, criterion, device):
    model.eval()

    total_loss = 0
    total_acc = 0
    total_simple_acc = 0

    start = time.time()

    for batch in dataloader:
        image = batch["image"].to(device)
        question = batch["question"].to(device)
        answers = batch["answers"].to(device)
        mode_answer = batch["mode_answer"].to(device)

        pred = model(image, question)

        if LOSS_TYPE == "hard":
            loss = criterion(pred, mode_answer)
        elif LOSS_TYPE == "soft":
            soft_target = build_soft_target(answers, pred.shape[1])
            loss = criterion(pred, soft_target)

        total_loss += loss.item()
        total_acc += VQA_criterion(pred.argmax(1), answers)
        total_simple_acc += (pred.argmax(1) == mode_answer).float().mean().item()

    n = len(dataloader)
    return total_loss / n, total_acc / n, total_simple_acc / n, time.time() - start


## 10. データの準備

学習データ・検証データ・テストデータを読み込む。辞書は学習データのものを検証・テストへ反映する。

In [ ]:
set_seed(SEED)

if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    raise RuntimeError(
        "GPU (CUDA/MPS) が利用できません。CPU 実行は許可されていません。"
    )
print("device =", device)

transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
])

train_dataset = VQADataset(
    df_path="./data/train_split.json",
    image_dir="./data/train",
    transform=transform,
)

valid_dataset = VQADataset(
    df_path="./data/valid_split.json",
    image_dir="./data/train",
    transform=transform,
)
valid_dataset.update_dict(train_dataset)

train_loader = torch.utils.data.DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=2, pin_memory=True,
)
valid_loader = torch.utils.data.DataLoader(
    valid_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=2, pin_memory=True,
)


## 11. 学習

検証 Acc が最良のモデルを `model.pt` として保存する。

In [ ]:
os.makedirs("./outputs/checkpoints", exist_ok=True)

model = VQAModel(
    vocab_size=len(train_dataset.question2idx) + 1,
    n_answer=len(train_dataset.answer2idx),
).to(device)

if LOSS_TYPE == "hard":
    criterion = nn.CrossEntropyLoss()
elif LOSS_TYPE == "soft":
    criterion = SoftCrossEntropyLoss()
else:
    raise ValueError(f"Unknown LOSS_TYPE: {LOSS_TYPE}")

if OPTIMIZER == "adam":
    optimizer = torch.optim.Adam(
        model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY,
    )
else:
    raise ValueError(f"Unknown OPTIMIZER: {OPTIMIZER}")

best_acc = -1.0

for epoch in range(NUM_EPOCHS):
    train_loss, train_acc, train_simple_acc, train_time = train_one_epoch(
        model, train_loader, optimizer, criterion, device,
    )
    valid_loss, valid_acc, valid_simple_acc, valid_time = validate(
        model, valid_loader, criterion, device,
    )
    print(
        f"Epoch [{epoch+1}/{NUM_EPOCHS}] "
        f"Train Acc={train_acc:.4f} Valid Acc={valid_acc:.4f}"
    )
    if valid_acc > best_acc:
        best_acc = valid_acc
        torch.save(model.state_dict(), "./model.pt")

print("best valid acc =", best_acc)


## 12. 推論 (テストデータ)

`data/valid.json` をテストデータとして予測し `submission.npy` を作成する。

In [ ]:
test_dataset = VQADataset(
    df_path="./data/valid.json",
    image_dir="./data/valid",
    transform=transform,
    answer=False,
)
test_dataset.update_dict(train_dataset)

test_loader = torch.utils.data.DataLoader(
    test_dataset, batch_size=1, shuffle=False,
    num_workers=2, pin_memory=True,
)

model.load_state_dict(torch.load("./model.pt", map_location=device))
model.eval()

submission = []
with torch.no_grad():
    for i, batch in enumerate(test_loader):
        if i % 500 == 0:
            print(f"{i}/{len(test_dataset)}")
        image = batch["image"].to(device)
        question = batch["question"].to(device)
        pred = model(image, question)
        pred = pred.argmax(1).item()
        submission.append(train_dataset.idx2answer[pred])

submission = np.array(submission)
np.save("./submission.npy", submission)
print("submission.npy saved:", submission.shape)


## 13. 提出ファイル作成

`submission.npy` / `model.pt` / 本 Notebook をまとめて zip 化する。
**注意**: `NOTEBOOK_NAME` を実際に保存している Notebook のファイル名に合わせること。

In [ ]:
# 本 Notebook のファイル名（保存名に合わせて変更）
NOTEBOOK_NAME = "DL_Basic_2026_Spring_competition_VQA.ipynb"

with ZipFile("submission.zip", "w") as zf:
    zf.write("submission.npy")
    zf.write("model.pt")
    if os.path.exists(NOTEBOOK_NAME):
        zf.write(NOTEBOOK_NAME)
    else:
        print(f"[警告] {NOTEBOOK_NAME} が見つかりません。"
              f" Notebook を保存後にこのセルを再実行してください。")

print("submission.zip created")
